# Extension 03 Minimal surfaces

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**Why is a uniform $q = 1$ net not a minimal soap film, and how do cotangent force densities find the minimal area?**

**Given:** two horizontal circular rings of radius $R = 3\text{ m}$ at $z = -1.6\text{ m}$ and $z = 1.6\text{ m}$, with zero external load ${\color{#2e9e4a} p = 0}$.

**Unknown:** the positions of the ${\color{#d62828} \text{free}}$ nodes on the soap film stretched between the two ${\color{#2a6fdb} \text{fixed}}$ rings.

**Equilibrium:** solving `# equation 6.32` with uniform $q = 1$ minimises the sum of squared bar lengths $\frac{1}{2}\sum l_e^2$, which pinches the waist too far inward. Setting each bar's force density from the opposite angles $\alpha_e, \beta_e$ of its two adjacent triangles,

$$ q_e = \frac{1}{2}\left(\cot\alpha_e + \cot\beta_e\right), \qquad \cot\theta = \frac{a \cdot b}{\|a \times b\|} $$

makes $\frac{1}{2}\sum q_e\,l_e^2$ equal the surface area at the current geometry, converging to Euler's minimal catenoid.

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_equilibrium.svg"
       style="width:100%; height:auto;"
       alt="Equilibrium of membrane tensions around an interior node" />
  <p style="text-align:center;">
    <em>Updating force densities from opposite triangle cotangents turns a uniform cable net into a minimal soap film.</em>
  </p>
</div>

**References:**

- **Mite**, https://github.com/jikempff/Mite

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Network

In [1]:
import numpy as np

# cylinder mesh between two coaxial rings of radius R = 3 m at z = -1.6 m and z = +1.6 m
n_ring, n_levels = 24, 9
R, H = 3.0, 1.6
angles = np.linspace(0.0, 2.0 * np.pi, n_ring, endpoint=False)
levels = np.linspace(-H, H, n_levels)

th_grid, z_grid = np.meshgrid(angles, levels)
x = (R * np.cos(th_grid)).flatten()
y = (R * np.sin(th_grid)).flatten()
z = z_grid.flatten()

# triangles and unique bars
tris = []
for j in range(n_levels - 1):
    for i in range(n_ring):
        i2 = (i + 1) % n_ring
        v00, v10 = j * n_ring + i, j * n_ring + i2
        v01, v11 = (j + 1) * n_ring + i, (j + 1) * n_ring + i2
        tris += [(v00, v10, v11), (v00, v11, v01)]

edges = sorted({(min(u, v), max(u, v)) for a, b, c in tris for u, v in ((a, b), (b, c), (c, a))})
edge_index = {e: k for k, e in enumerate(edges)}

C = np.zeros((len(edges), len(x)))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1, -1

# bottom and top rings are fixed, middle rings are free
fixed = list(range(n_ring)) + list(range((n_levels - 1) * n_ring, n_levels * n_ring))
free  = [k for k in range(len(x)) if k not in fixed]
C_N, C_F = C[:, free], C[:, fixed]

# Uniform Force Density Versus Cotangent Iteration

First solve once with $q = 1$. Then update $q_e = \frac{1}{2}(\cot\alpha_e + \cot\beta_e)$ from the current triangles and repeat `# equation 6.32`.

In [2]:
def surface_area(x, y, z):
    pts = np.column_stack([x, y, z])
    p0 = pts[[t[0] for t in tris]]
    p1 = pts[[t[1] for t in tris]]
    p2 = pts[[t[2] for t in tris]]
    return 0.5 * np.sum(np.linalg.norm(np.cross(p1 - p0, p2 - p0), axis=1))


def cotangent_q(x, y, z):
    pts = np.column_stack([x, y, z])
    q = np.zeros(len(edges))
    for i0, i1, i2 in tris:
        tri = (i0, i1, i2)
        for m in range(3):
            a, b, opp = tri[m], tri[(m + 1) % 3], tri[(m + 2) % 3]
            u, v = pts[a] - pts[opp], pts[b] - pts[opp]
            area2 = np.linalg.norm(np.cross(u, v))
            if area2 > 1e-10:
                q[edge_index[(min(a, b), max(a, b))]] += 0.5 * np.dot(u, v) / area2
    return np.maximum(q, 1e-4)


# 1. uniform q = 1 solve
q = np.ones(len(edges))
Q = np.diag(q)
x_uni, y_uni, z_uni = x.copy(), y.copy(), z.copy()

# equation 6.32
A = C_N.T @ Q @ C_N
x_uni[free] = np.linalg.solve(A, -C_N.T @ Q @ C_F @ x_uni[fixed])
y_uni[free] = np.linalg.solve(A, -C_N.T @ Q @ C_F @ y_uni[fixed])
z_uni[free] = np.linalg.solve(A, -C_N.T @ Q @ C_F @ z_uni[fixed])

# 2. cotangent minimal surface loop
x_min, y_min, z_min = x_uni.copy(), y_uni.copy(), z_uni.copy()
for step in range(25):
    q = cotangent_q(x_min, y_min, z_min)
    Q = np.diag(q)

    # equation 6.32
    A = C_N.T @ Q @ C_N
    x_min[free] = np.linalg.solve(A, -C_N.T @ Q @ C_F @ x_min[fixed])
    y_min[free] = np.linalg.solve(A, -C_N.T @ Q @ C_F @ y_min[fixed])
    z_min[free] = np.linalg.solve(A, -C_N.T @ Q @ C_F @ z_min[fixed])

waist = slice((n_levels // 2) * n_ring, (n_levels // 2 + 1) * n_ring)
print("Euler exact catenoid neck radius = 2.461 m")
print("uniform q = 1 waist radius       =", round(np.hypot(x_uni[waist], y_uni[waist]).mean(), 3), "m  area =", round(surface_area(x_uni, y_uni, z_uni), 2), "m^2")
print("cotangent minimal waist radius   =", round(np.hypot(x_min[waist], y_min[waist]).mean(), 3), "m  area =", round(surface_area(x_min, y_min, z_min), 2), "m^2")

Euler exact catenoid neck radius = 2.461 m
uniform q = 1 waist radius       = 1.804 m  area = 58.93 m^2
cotangent minimal waist radius   = 2.45 m  area = 56.91 m^2


# Viewer

Gray: uniform $q = 1$ net pinched inward. Black: minimal catenoid found with cotangent force densities.

In [3]:
from session_py import Color, Matrix, Session
from session_jupyter import show

# the colours of the figures: free nodes red, fixed nodes blue, reference grey
RED  = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)
GRAY = Color(0.70, 0.70, 0.70, 1.0)

f = q * np.sqrt((C @ x_min)**2 + (C @ y_min)**2 + (C @ z_min)**2)
colors = [Color(0.60 - 0.60 * fv / f.max(), 0.60 - 0.60 * fv / f.max(), 0.60 - 0.60 * fv / f.max(), 1.0) for fv in f]

session = Session("fdm_ext03_catenoid")
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x_uni, y_uni, z_uni]).tolist()), [GRAY] * len(edges), [1.3] * len(edges))
session.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(np.column_stack([x_min, y_min, z_min]).tolist()), colors, [2.6] * len(edges))

for net in range(2):
    for k in range(len(x)):
        pt = session.objects.points[net * len(x) + k]
        pt.pointcolor = BLUE if k in fixed else RED
        pt.width = 10.0 if k in fixed else 5.5

show(session, "fdm_ext03_catenoid")

# Observation

Uniform $q = 1$ depends on how the cylinder is meshed and over pinches the neck to $1.80\text{ m}$. Updating $q$ with triangle cotangents lowers the total area and lands directly on Euler's catenoid neck of $2.46\text{ m}$.